# Synthetic Data Pipeline

This notebook runs the synthetic data generation pipeline.

### Before getting started:
- Ensure you have read `docs/*` and `README.md`
- Check that `params.py` and `config.py` are correct.
- Check the `call_LLM` and `red_write_data` functions in `processing.py` are correctly configured for your platform .
- Check that your input data exists and is correctly formatted.

First, import the required classes. 

In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [2]:
from src.data_generator import generate_patients, generate_admissions, generate_journeys, generate_clinical_notes, add_augmentations, save_final_outputs
from datetime import datetime
from config.params import PARAMS

In [3]:
# INSERT RUN NAME BELOW
# This will be saved in the journey dataset at the end of the notebook for evaluation purposes
run_name = "trial"
current_time = datetime.now()
version_tag = current_time.strftime("%Y-%m-%d") + f"/{run_name}"
print(f"Run name: {run_name}\nDate: {current_time}\nVersion tag: {version_tag}")

Run name: trial
Date: 2026-11-18 19:02:44.779245
Version tag: 2026-11-18/trial


In [4]:
output_saver = save_final_outputs()
output_saver.run(run_name, current_time, version_tag)

Preparing data... Saving Data... Writing to patients_output dataset
Successfully wrote to patients_output dataset
Writing to admissions dataset
Successfully wrote to admissions dataset
Writing to encounters dataset
Successfully wrote to encounters dataset
Writing to journey_metrics dataset
Successfully wrote to journey_metrics dataset
Writing to clinical_notes dataset
Successfully wrote to clinical_notes dataset
Writing to journeys dataset
Successfully wrote to journeys dataset
DONE


## 1 - Get Patients Information and Admissions

- Generates a list of patients with corresponding admission reasons.


In [11]:
patient_generator = generate_patients()
patients = await patient_generator.run(return_output = True)
patient_generator.write_patients_to_dataset()

Sampling 15 NSDUH patient profiles... DONE


In [12]:
admission_generator = generate_admissions()
admissions = await admission_generator.run(return_output = True)
admission_generator.write_admissions_to_dataset()

Generating 15 admissions... DONE


## 2 - Generating and Filtering Journeys

- Generates, validates, and adds details to patient journeys.
- Filtering removes any document types that are not listed as possible event types in `params.py`.

In [13]:
journey_generator = generate_journeys()
journeys = await journey_generator.run(return_outputs = True)
journey_generator.write_journeys_to_dataset()

Generating simple journeys... Validating simple journeys...
Validator Changes:
 {'Patient_0': 1, 'Patient_1': 1, 'Patient_2': 0, 'Patient_3': 1, 'Patient_4': 1, 'Patient_5': 1, 'Patient_6': 1, 'Patient_7': 1, 'Patient_8': 1, 'Patient_9': 1, 'Patient_10': 1, 'Patient_11': 1, 'Patient_12': 1, 'Patient_13': 1, 'Patient_14': 1}
Generating extra details... Generating staff personas... Creating full detailed journeys... Filtering journeys...
Patient 0 - Removing events: ['emergency_admission']
Patient 1 - Removing events: ['emergency_admission']
Patient 2 - Removing events: []
Patient 3 - Removing events: ['emergency_admission']
Patient 4 - Removing events: []
Patient 5 - Removing events: ['post-operative ward review']
Patient 6 - Removing events: ['emergency_admission']
Patient 7 - Removing events: ['emergency_admission']
Patient 8 - Removing events: ['orthopaedic admission review']
Patient 9 - Removing events: ['emergency_admission']
Patient 10 - Removing events: ['emergency_admission']
Pa

## 3 - Generate and Validate Clinical Notes

- Uses LLMs to generate clinical notes. 
- Validates each note using an LLM Judge. 

In [14]:
clinical_note_generator = generate_clinical_notes()
notes = await clinical_note_generator.run(return_output = True)
clinical_note_generator.write_patient_documents_to_dataset()

Patient 0: Generating Notes... Error cleaning with LLM: Expecting value: line 1 column 2 (char 1), returning raw_output: o
Error cleaning with LLM: Expecting value: line 1 column 2 (char 1), returning raw_output: n
Error cleaning with LLM: Expecting value: line 1 column 2 (char 1), returning raw_output: r
Error cleaning with LLM: Expecting value: line 1 column 2 (char 1), returning raw_output: c
Error cleaning with LLM: Expecting value: line 1 column 2 (char 1), returning raw_output: h
Error cleaning with LLM: Expecting value: line 1 column 2 (char 1), returning raw_output: n
Error cleaning with LLM: Expecting value: line 1 column 2 (char 1), returning raw_output: e
Error cleaning with LLM: Expecting value: line 1 column 2 (char 1), returning raw_output: l
Error cleaning with LLM: Expecting value: line 1 column 2 (char 1), returning raw_output: n
Error cleaning with LLM: Expecting value: line 1 column 2 (char 1), returning raw_output: a
Error cleaning with LLM: Expecting value: line 1 

In [15]:
import json
print(json.loads(notes[0][7]))

{'note_subject': 'Respiratory ward nursing obs', 'note_type': 'Nursing', 'Content': '01/01/26 14:00. Respiratory ward. Settled in bed. Comfortable at rest; denies chest pain. SpO₂ 94% on O₂ 2 L/min via nasal cannula. RR 20/min, pulse 102 bpm, BP 126/78 mmHg, temperature 36.7°C. No medication changes were made. Maintain O₂ at 2 L/min via nasal cannula pending medical review. Documented by Nurse Gary Albert Perkins (Staff Nurse).'}


## 4 - Add Augmentations to clinical Notes

This section allows for the augmentatio of clinical notes by:

- Replacing long phrases with abbreviations.
- Adding typos.
- Adding signatures.

In [16]:
augmentator = add_augmentations()
augmented_notes = await augmentator.run(True)
augmentator.write_final_documents_to_dataset()

Patient 0
Note: 0: Added (Estimated) 1 Abbreviations. Added 1 typos.
Note: 1: Added (Estimated) 1 Abbreviations. Added 1 typos.
Note: 2: Added (Estimated) 3 Abbreviations. Added 1 typos.
Note: 3: Added (Estimated) 4 Abbreviations. Added 1 typos.
Note: 4: Added (Estimated) 4 Abbreviations. Added 2 typos.
Note: 5: Added (Estimated) 19 Abbreviations. Added 5 typos.
Note: 6: Added (Estimated) 9 Abbreviations. Added 4 typos.
Note: 7: Added (Estimated) 3 Abbreviations. Added 2 typos.
Note: 8: Added (Estimated) 4 Abbreviations. Added 1 typos.
Note: 9: Added (Estimated) 3 Abbreviations. Added 0 typos.
Note: 10: Added (Estimated) 1 Abbreviations. Added 1 typos.
Note: 11: Added (Estimated) 9 Abbreviations. Added 0 typos.
Note: 12: Added (Estimated) 1 Abbreviations. Added 2 typos.
Note: 13: Added (Estimated) 4 Abbreviations. Added 1 typos.
Note: 14: Added (Estimated) 2 Abbreviations. Added 1 typos.
Note: 15: Added (Estimated) 0 Abbreviations. Added 0 typos.
Note: 16: Added (Estimated) 0 Abbreviat

## 5 - Write Clinical Notes to Dataset

- Writes the clinical notes to a dataset, alongside the patient journey and admission details.

In [4]:
output_saver = save_final_outputs()
output_saver.run(run_name, current_time, version_tag)

Preparing data... 

/work/pi_hongyu_umass_edu/raymondx/poulett_synthetic_clinical_notes/src/data_generator.py:623: SyntaxWarning: invalid escape sequence '\s'
  df["ChiefComplaintDescription"] = df["ChiefComplaintDescription"].str.split("\s+\(").str.get(0)
/work/pi_hongyu_umass_edu/raymondx/poulett_synthetic_clinical_notes/src/data_generator.py:624: SyntaxWarning: invalid escape sequence '\s'
  df["DiagnosisDescription"] = df["DiagnosisDescription"].str.split("\s+\(").str.get(0)


ValueError: Unrecognized datetime format: 08:35